In [2]:
# download_data the dataset
import pandas as pd
import requests
import os
import psycopg2
from psycopg2.extras import execute_batch
from datetime import datetime

os.makedirs('data', exist_ok=True)

url = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries.csv"
print("Downloading FEMA disaster data...")
response = requests.get(url)

with open('data/fema_disasters.csv', 'wb') as f:
    f.write(response.content)

print("Downloaded successfully!")

Downloaded successfully!


In [3]:
# explore the dataset


df = pd.read_csv('data/fema_disasters.csv', low_memory=False)

print(f"Total records:  {len(df):,}")
print(f"Columns:        {df.columns.tolist()}")
print(f"\nDisaster types:\n{df['incidentType'].value_counts().head(10)}")
print(f"\nTop states:\n{df['state'].value_counts().head(10)}")
print(f"\nDate range: {df['declarationDate'].min()} to {df['declarationDate'].max()}")

Total records:  70,094
Columns:        ['femaDeclarationString', 'disasterNumber', 'state', 'declarationType', 'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle', 'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared', 'hmProgramDeclared', 'incidentBeginDate', 'incidentEndDate', 'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'designatedArea', 'declarationRequestNumber', 'lastIAFilingDate', 'incidentId', 'region', 'designatedIncidentTypes', 'lastRefresh', 'hash', 'id']

Disaster types:
incidentType
Severe Storm        19338
Hurricane           13726
Flood               11380
Biological           7857
Fire                 3903
Snowstorm            3707
Severe Ice Storm     2956
Tornado              1623
Winter Storm         1378
Drought              1292
Name: count, dtype: int64

Top states:
state
TX    5424
KY    3376
MO    2840
FL    2794
GA    2768
VA    2756
LA    2677
OK    2593
NC    2431
MS    2134
Name: count, dty

In [7]:
# load data
conn = psycopg2.connect(
    host="localhost", port=5432,
    user="postgres", password="654321",
    database="disaster_db"
)
cursor = conn.cursor()

print("Loading FEMA disaster data...")
df = pd.read_csv('data/fema_disasters.csv', low_memory=False)

# Clean and select relevant columns
df = df[[
    'disasterNumber', 'incidentType', 'declarationDate',
    'incidentBeginDate', 'incidentEndDate',
    'state', 'designatedArea', 'declarationRequestNumber'
]].copy()

df.columns = [
    'disaster_id', 'disaster_type', 'declaration_date',
    'incident_begin', 'incident_end',
    'state', 'county', 'declaration_type'
]

# Parse dates
df['declaration_date'] = pd.to_datetime(df['declaration_date'], errors='coerce')
df['incident_begin']   = pd.to_datetime(df['incident_begin'],   errors='coerce')
df['incident_end']     = pd.to_datetime(df['incident_end'],     errors='coerce')

# Add derived columns
df['year']       = df['declaration_date'].dt.year
df['month']      = df['declaration_date'].dt.month
df['month_name'] = df['declaration_date'].dt.strftime('%b')
df['quarter']    = df['declaration_date'].dt.quarter

# Add region
region_map = {
    'AL':'South','AR':'South','FL':'South','GA':'South',
    'KY':'South','LA':'South','MS':'South','NC':'South',
    'SC':'South','TN':'South','VA':'South','WV':'South',
    'CT':'Northeast','MA':'Northeast','ME':'Northeast',
    'NH':'Northeast','NJ':'Northeast','NY':'Northeast',
    'PA':'Northeast','RI':'Northeast','VT':'Northeast',
    'IL':'Midwest','IN':'Midwest','IA':'Midwest',
    'KS':'Midwest','MI':'Midwest','MN':'Midwest',
    'MO':'Midwest','NE':'Midwest','ND':'Midwest',
    'OH':'Midwest','SD':'Midwest','WI':'Midwest',
    'AK':'West','AZ':'West','CA':'West','CO':'West',
    'HI':'West','ID':'West','MT':'West','NM':'West',
    'NV':'West','OR':'West','UT':'West','WA':'West',
    'WY':'West','TX':'South','OK':'South'
}
df['region'] = df['state'].map(region_map).fillna('Other')

# Remove nulls
df = df[df['declaration_date'].notna()]
df = df[df['disaster_type'].notna()]

print(f"Loading {len(df):,} records...")

rows = [tuple(
    None if pd.isna(v) else
    (v.date() if hasattr(v, 'date') else v)
    for v in row
) for row in df[[
    'disaster_id','disaster_type','declaration_date',
    'incident_begin','incident_end','state','county',
    'region','declaration_type','year','month',
    'month_name','quarter'
]].values]

execute_batch(cursor, """
    INSERT INTO gold.disasters (
        disaster_id, disaster_type, declaration_date,
        incident_begin, incident_end, state, county,
        region, declaration_type, year, month,
        month_name, quarter
    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", rows, page_size=500)

conn.commit()
cursor.close()
conn.close()
print("Done! Ready for Power BI.")

Loading FEMA disaster data...
Loading 70,094 records...
Done! Ready for Power BI.
